In [1]:
# Install CatBoost if not already installed
!pip install catboost -q

import pandas as pd
import numpy as np
import os
import warnings
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Import Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

# Configuration
warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Libraries imported successfully.


In [2]:
# Initialize file paths
train_path = ''
test_path = ''
sub_path = ''

# Automatically search for dataset files in Kaggle directories
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if 'train.csv' in filename:
            train_path = full_path
        elif 'test.csv' in filename:
            test_path = full_path
        elif 'sample_submission.csv' in filename:
            sub_path = full_path

# Load data into DataFrames
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

print(f"Data loaded. Train shape: {train.shape}, Test shape: {test.shape}")

Data loaded. Train shape: (700000, 26), Test shape: (300000, 25)


In [3]:
# 1. Separate IDs and Target variable
test_ids = test['id']
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

target = train['diagnosed_diabetes']
train = train.drop('diagnosed_diabetes', axis=1)

# 2. Combine Train and Test for consistent One-Hot Encoding
train_len = len(train)
combined = pd.concat([train, test], axis=0)

# 3. One-Hot Encoding
combined_encoded = pd.get_dummies(combined, drop_first=True)

# 4. Split back into Train and Test sets
X = combined_encoded.iloc[:train_len]
X_test = combined_encoded.iloc[train_len:]

# 5. Standardization (Scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Convert arrays back to DataFrames for easier handling
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Preprocessing complete.")

Preprocessing complete.


In [4]:
# Configuration for Cross-Validation
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to store Out-Of-Fold (OOF) predictions
xgb_oof = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))

# Arrays to store Test predictions
xgb_test_pred = np.zeros(len(X_test))
lgbm_test_pred = np.zeros(len(X_test))
cat_test_pred = np.zeros(len(X_test))

print(f"Starting {N_SPLITS}-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, target)):
    print(f"Processing Fold {fold + 1}/{N_SPLITS}...")
    
    # Split data for the current fold
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = target.iloc[train_idx], target.iloc[val_idx]
    
    # --- 1. XGBoost ---
    xgb = XGBClassifier(
        n_estimators=2000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, n_jobs=-1, early_stopping_rounds=100
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    
    xgb_oof[val_idx] = xgb.predict_proba(X_val)[:, 1]
    xgb_test_pred += xgb.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

    # --- 2. LightGBM ---
    lgbm = LGBMClassifier(
        n_estimators=2000, learning_rate=0.015, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, metric='auc',
        random_state=42, n_jobs=-1, verbosity=-1
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
             callbacks=[early_stopping(100, verbose=False)])
    
    lgbm_oof[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    lgbm_test_pred += lgbm.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

    # --- 3. CatBoost ---
    cat = CatBoostClassifier(
        iterations=2000, learning_rate=0.015, depth=6,
        eval_metric='AUC', random_seed=42, verbose=0,
        early_stopping_rounds=100, allow_writing_files=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    cat_oof[val_idx] = cat.predict_proba(X_val)[:, 1]
    cat_test_pred += cat.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

print("Phase 1 training complete.")

Starting 5-Fold Cross-Validation...
Processing Fold 1/5...
Processing Fold 2/5...
Processing Fold 3/5...
Processing Fold 4/5...
Processing Fold 5/5...
Phase 1 training complete.


In [5]:
# Create DataFrames for optimization
oof_preds = pd.DataFrame({'xgb': xgb_oof, 'lgbm': lgbm_oof, 'cat': cat_oof})
test_preds = pd.DataFrame({'xgb': xgb_test_pred, 'lgbm': lgbm_test_pred, 'cat': cat_test_pred})

# Define the function to minimize (Negative AUC)
def minimize_auc(weights):
    weights = np.array(weights)
    # Normalize weights so they sum to 1
    if weights.sum() == 0: return 0
    weights /= weights.sum()
    
    final_oof = (oof_preds['xgb'] * weights[0]) + \
                (oof_preds['lgbm'] * weights[1]) + \
                (oof_preds['cat'] * weights[2])
    
    return -roc_auc_score(target, final_oof)

# Perform optimization
print("Optimizing ensemble weights...")
initial_weights = [0.33, 0.33, 0.33]
result = minimize(minimize_auc, initial_weights, method='Nelder-Mead')

best_weights = result.x / result.x.sum()
print(f"Optimal Weights -> XGB: {best_weights[0]:.4f}, LGBM: {best_weights[1]:.4f}, Cat: {best_weights[2]:.4f}")
print(f"Optimized CV Score: {-result.fun:.5f}")

# Calculate Phase 1 Final Prediction
phase1_pred = (test_preds['xgb'] * best_weights[0]) + \
              (test_preds['lgbm'] * best_weights[1]) + \
              (test_preds['cat'] * best_weights[2])

Optimizing ensemble weights...
Optimal Weights -> XGB: 0.5801, LGBM: 0.9606, Cat: -0.5407
Optimized CV Score: 0.72686


In [6]:
# Thresholds for pseudo labeling
HIGH_THRESHOLD = 0.95  # Confident positive
LOW_THRESHOLD = 0.05   # Confident negative

# Create a copy of test data for augmentation
X_test_pseudo = X_test_scaled.copy()
X_test_pseudo['pseudo_target'] = phase1_pred

# Filter confident predictions
pseudo_high = X_test_pseudo[X_test_pseudo['pseudo_target'] >= HIGH_THRESHOLD]
pseudo_high['diagnosed_diabetes'] = 1

pseudo_low = X_test_pseudo[X_test_pseudo['pseudo_target'] <= LOW_THRESHOLD]
pseudo_low['diagnosed_diabetes'] = 0

# Concatenate high and low confidence samples
pseudo_data = pd.concat([pseudo_high, pseudo_low], axis=0).drop('pseudo_target', axis=1)

# Reconstruct original training set to match columns
X_train_full = pd.concat([X_scaled, target], axis=1)

# Ensure column order matches
pseudo_data = pseudo_data[X_train_full.columns]

# Create the new augmented training set
new_train = pd.concat([X_train_full, pseudo_data], axis=0)
new_X = new_train.drop('diagnosed_diabetes', axis=1)
new_y = new_train['diagnosed_diabetes']

print(f"Pseudo Labeling: Added {len(pseudo_data)} samples to training set.")
print(f"New training set shape: {new_X.shape}")

Pseudo Labeling: Added 2400 samples to training set.
New training set shape: (702400, 36)


In [7]:
# Reset variables for Phase 2
pseudo_test_preds = np.zeros(len(X_test))

print("Starting Phase 2 (Retraining with Pseudo Labels)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(new_X, new_y)):
    print(f"Retraining Fold {fold + 1}/{N_SPLITS}...")
    
    X_tr, X_val = new_X.iloc[train_idx], new_X.iloc[val_idx]
    y_tr, y_val = new_y.iloc[train_idx], new_y.iloc[val_idx]
    
    # 1. XGBoost Retrain
    xgb = XGBClassifier(
        n_estimators=2000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, n_jobs=-1, early_stopping_rounds=100
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    p1 = xgb.predict_proba(X_test_scaled)[:, 1]

    # 2. LightGBM Retrain
    lgbm = LGBMClassifier(
        n_estimators=2000, learning_rate=0.015, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, metric='auc',
        random_state=42, n_jobs=-1, verbosity=-1
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
             callbacks=[early_stopping(100, verbose=False)])
    p2 = lgbm.predict_proba(X_test_scaled)[:, 1]

    # 3. CatBoost Retrain
    cat = CatBoostClassifier(
        iterations=2000, learning_rate=0.015, depth=6,
        eval_metric='AUC', random_seed=42, verbose=0,
        early_stopping_rounds=100, allow_writing_files=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    p3 = cat.predict_proba(X_test_scaled)[:, 1]

    # Ensemble using best weights found in Phase 1
    fold_pred = (p1 * best_weights[0]) + \
                (p2 * best_weights[1]) + \
                (p3 * best_weights[2])
    
    pseudo_test_preds += fold_pred / N_SPLITS

print("Phase 2 retraining complete.")

Starting Phase 2 (Retraining with Pseudo Labels)...
Retraining Fold 1/5...
Retraining Fold 2/5...
Retraining Fold 3/5...
Retraining Fold 4/5...
Retraining Fold 5/5...
Phase 2 retraining complete.


In [8]:
# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': pseudo_test_preds
})

# Save to CSV
submission_df.to_csv('submission.csv', index=False)
print("submission.csv(v5) created successfully.")

# Display first few rows
display(submission_df.head())

submission.csv(v5) created successfully.


,id,diagnosed_diabetes
0,700000,0.490880
1,700001,0.697172
2,700002,0.765941
3,700003,0.379688
4,700004,0.918030
